# Katthammarsvik coastal vegetation mapping
**Sentinel-2 → sun-glint correction → classification → GeoJSON (SWEREF 99 TM)**

Emma Kathleen Larsen · Visby, Gotland · emma@geoteka.se

This notebook runs the full pipeline on a Sentinel-2 L2A scene covering
Katthammarsvik. The water here is turbid (calcareous sediment) and glint-prone,
so the deglinting step is what makes the optical signal usable.

## 1. Setup
The processing logic lives in `src/` so it can be reused and tested outside the notebook.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src import io_s2, deglint, classify

# --- point this at your downloaded Sentinel-2 L2A scene (.SAFE folder) ---
SCENE_DIR = "../data/S2_katthammarsvik.SAFE"
OUT_GEOJSON = "../outputs/katthammarsvik_vegetation.geojson"


## 2. Load the scene
Blue/green/red/NIR as reflectance plus a validity mask from the SCL layer.

In [ ]:
scene = io_s2.load_scene(SCENE_DIR)
red, green, nir = scene['red'], scene['green'], scene['nir']
print('grid:', red.shape, '| valid pixels:', int(scene['valid'].sum()))


## 3. Sun glint correction (Hedley et al. 2005)
Fit the glint regression over optically deep water, then subtract per pixel.

In [ ]:
deep = classify.deepwater_mask(green, nir, scene['valid'])
before = scene['red'].copy()
scene = deglint.deglint_scene(scene, deep)
after = scene['red']

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, img, t in zip(ax, (before, after), ('Red — raw', 'Red — deglinted')):
    a.imshow(img, vmin=0, vmax=0.15, cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()


## 4. Water mask + vegetation classification
NDWI separates water from land; NDVI flags vegetation inside the water/coastal zone.

In [ ]:
water = classify.water_mask(green, nir, scene['valid'], ndwi_thresh=0.0)
classes = classify.classify_vegetation(scene, water, ndvi_thresh=0.10)

plt.figure(figsize=(7, 7))
plt.imshow(classes, cmap='viridis')
plt.title('0=unclassified  1=vegetation  2=open water'); plt.axis('off'); plt.show()
print('vegetation pixels:', int((classes == 1).sum()))


## 5. Vectorise → GeoJSON in SWEREF 99 TM
Output drops straight into a Lantmäteriet-aligned QGIS project (EPSG:3006).

In [ ]:
gdf = classify.to_geojson(classes, scene['profile'], OUT_GEOJSON)
print(f'{len(gdf)} polygons → {OUT_GEOJSON} ({classify.SWEREF99_TM})')
gdf.head()


## Notes
- Thresholds (`ndwi_thresh`, `ndvi_thresh`) are tuned for this site and must be
  re-checked per scene.
- Validation against field points is partial — thesis-scale result, not an
  operational product.